<a href="https://colab.research.google.com/github/salochaud/DatosMasivos/blob/main/Tarea_3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark -q

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round, when

spark = SparkSession.builder \
    .appName("Tarea 3 - FIFA DataFrames") \
    .getOrCreate()

In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/players_22.csv")

df.show(5)

+---------+--------------------+-----------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+------------+-------------------+--------------------+------------+-------------+------------------+----------------+-----------+-------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-----------+---------+---------+------------------+--------------------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------------------+----------

In [4]:
df = df.withColumnRenamed("short_name", "nombre") \
       .withColumnRenamed("nationality_name", "pais") \
       .withColumnRenamed("club_name", "club")

df.select("nombre", "pais", "club", "overall", "potential", "age").show(5)

+-----------------+---------+-------------------+-------+---------+---+
|           nombre|     pais|               club|overall|potential|age|
+-----------------+---------+-------------------+-------+---------+---+
|         L. Messi|Argentina|Paris Saint-Germain|     93|       93| 34|
|   R. Lewandowski|   Poland|  FC Bayern München|     92|       92| 32|
|Cristiano Ronaldo| Portugal|  Manchester United|     91|       91| 36|
|        Neymar Jr|   Brazil|Paris Saint-Germain|     91|       91| 29|
|     K. De Bruyne|  Belgium|    Manchester City|     91|       91| 30|
+-----------------+---------+-------------------+-------+---------+---+
only showing top 5 rows


In [5]:
df = df.withColumn(
    "indice_tecnico",
    round((col("pace") + col("shooting") + col("passing") + col("dribbling")) / 4, 2)
)

df.select("nombre", "overall", "pace", "shooting", "passing", "dribbling", "indice_tecnico").show(10)

+-----------------+-------+----+--------+-------+---------+--------------+
|           nombre|overall|pace|shooting|passing|dribbling|indice_tecnico|
+-----------------+-------+----+--------+-------+---------+--------------+
|         L. Messi|     93|  85|      92|     91|       95|         90.75|
|   R. Lewandowski|     92|  78|      92|     79|       86|         83.75|
|Cristiano Ronaldo|     91|  87|      94|     80|       88|         87.25|
|        Neymar Jr|     91|  91|      83|     86|       94|          88.5|
|     K. De Bruyne|     91|  76|      86|     93|       88|         85.75|
|         J. Oblak|     91|NULL|    NULL|   NULL|     NULL|          NULL|
|        K. Mbappé|     91|  97|      88|     80|       92|         89.25|
|         M. Neuer|     90|NULL|    NULL|   NULL|     NULL|          NULL|
|    M. ter Stegen|     90|NULL|    NULL|   NULL|     NULL|          NULL|
|          H. Kane|     90|  70|      91|     83|       83|         81.75|
+-----------------+------

In [6]:
df = df.withColumn(
    "categoria",
    when(col("overall") >= 85, "Mundial")
    .when(col("overall") >= 75, "Bueno")
    .when(col("overall") >= 65, "Regular")
    .otherwise("Bajo")
)

df.select("nombre", "overall", "categoria").show(10)

+-----------------+-------+---------+
|           nombre|overall|categoria|
+-----------------+-------+---------+
|         L. Messi|     93|  Mundial|
|   R. Lewandowski|     92|  Mundial|
|Cristiano Ronaldo|     91|  Mundial|
|        Neymar Jr|     91|  Mundial|
|     K. De Bruyne|     91|  Mundial|
|         J. Oblak|     91|  Mundial|
|        K. Mbappé|     91|  Mundial|
|         M. Neuer|     90|  Mundial|
|    M. ter Stegen|     90|  Mundial|
|          H. Kane|     90|  Mundial|
+-----------------+-------+---------+
only showing top 10 rows


In [7]:
jugadores_completos = df.filter(
    (col("indice_tecnico") >= 80) &
    (col("defending") >= 70) &
    (col("physic") >= 70)
)

jugadores_completos.select("nombre", "club", "pais", "overall", "indice_tecnico", "defending", "physic").show()

+---------------+-------------------+-----------+-------+--------------+---------+------+
|         nombre|               club|       pais|overall|indice_tecnico|defending|physic|
+---------------+-------------------+-----------+-------+--------------+---------+------+
|Bruno Fernandes|  Manchester United|   Portugal|     88|          83.5|       70|    77|
|    L. Goretzka|  FC Bayern München|    Germany|     87|         82.25|       81|    86|
|     F. de Jong|       FC Barcelona|Netherlands|     87|         80.75|       77|    78|
|   João Cancelo|    Manchester City|   Portugal|     86|         80.75|       80|    72|
|Marcos Llorente| Atlético de Madrid|      Spain|     86|         82.75|       78|    82|
|      A. Hakimi|Paris Saint-Germain|    Morocco|     85|         81.75|       76|    78|
|   T. Hernández|           AC Milan|     France|     84|         80.25|       77|    82|
|    J. Cuadrado|           Juventus|   Colombia|     83|         82.25|       76|    72|
+---------

In [8]:
ranking = jugadores_completos.groupBy("pais").count().orderBy(col("count").desc())
ranking.filter(col("count") > 1).show()

+--------+-----+
|    pais|count|
+--------+-----+
|Portugal|    2|
+--------+-----+

